In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/vi-sub-correction')
CKPT_ROOT  = DRIVE_ROOT / 'checkpoints'
DATA_DRIVE = DRIVE_ROOT / 'data'
for d in (CKPT_ROOT, DATA_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/vi-sub-corretion')
DATA_DIR = REPO_DIR / 'data' / 'synthetic'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Da ket noi Drive.')
print('Checkpoint ben vung :', CKPT_ROOT)
print('Du lieu tam (/content):', DATA_DIR)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

if not REPO_DIR.exists():
    !git clone https://github.com/lhhoang5425/vi-sub-corretion.git /content/vi-sub-corretion

%cd /content/vi-sub-corretion
!git pull
!pip install -r requirements.txt -q
!pip install -q openpyxl

print('--- 3 commit gan nhat ma Colab dang chay ---')
!git log --oneline -3

In [ ]:
import json
import re
from datasets import load_dataset
from src.data_processing.corpus_builder import build_noisy_corpus, format_stats, write_jsonl

# ĐƯỜNG B: vẫn 100,000 câu như Lần 4, nhưng là 100,000 câu SẠCH.
# Nguồn ShynBui trùng lặp nội bộ ~21% nên phải nạp dư rồi cắt sau khi khử trùng lặp.
NUM_TRAIN_CLEAN = 100_000   # số câu sạch mong muốn (sau khử trùng lặp)
NUM_FETCH = 220_000         # nạp dư để bù phần bị loại
SEED = 42
VARIANTS = ['legacy', 'asr_hard']
VAL_RATIO = 0.1

raw = load_dataset('ShynBui/Vietnamese_spelling_error', split=f'train[:{NUM_FETCH}]')
clean = [(r['text'] or '').strip() for r in raw]

# ── SỬA P1 (rò rỉ val↔train 20.8% của Lần 4) ─────────────────────────────────
# Lần 4 chia tập theo VỊ TRÍ DÒNG (i % 10). Dataset nguồn trùng lặp ~21% nên cùng
# một câu có bản vào train, bản vào val -> 1.894/9.121 câu val đã nằm trong train.
# corpus_builder khử trùng lặp trước, rồi chia theo HASH NỘI DUNG. Hàm này tự chặn
# và báo lỗi nếu còn rò rỉ, nên không thể tái diễn mà không biết.
corpus_by_profile = {
    p: build_noisy_corpus(clean, profile=p, val_ratio=VAL_RATIO, seed=SEED, max_clean=NUM_TRAIN_CLEAN)
    for p in VARIANTS
}
print(format_stats(corpus_by_profile[VARIANTS[0]]['stats']))
print('  (phần kiểm toán trên chỉ phụ thuộc nội dung, giống nhau cho mọi hồ sơ nhiễu)\n')

TONE = re.compile(r'[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]', re.I)
gold = [json.loads(l) for l in open('data/gold_test/gd1_basic/sample_test.jsonl', encoding='utf-8') if l.strip()]
gold_lost = sum(1 for g in gold if not TONE.search(g['noisy'])) / len(gold) * 100
print(f'ĐỀ THI yêu cầu khôi phục dấu thanh cho {gold_lost:.0f}% câu ({len(gold)} mẫu)\n')

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'{"HỒ SƠ NHIỄU":<12}{"train":>9}{"val":>8}{"% câu MẤT TRẮNG dấu":>23}   đánh giá')
print('-' * 74)
for prof in VARIANTS:
    corpus = corpus_by_profile[prof]
    n_tr = write_jsonl(corpus['train'], DATA_DIR / f'train_{prof}.jsonl')
    n_va = write_jsonl(corpus['validation'], DATA_DIR / f'val_{prof}.jsonl')
    rows = corpus['validation'][:500]
    lost = sum(1 for r in rows if not TONE.search(r['noisy'])) / len(rows) * 100
    tag = '<< KHỚP đề thi' if lost > 60 else '>> LỆCH đề thi (đây là lỗi Lần 4)'
    print(f'{prof:<12}{n_tr:>9,}{n_va:>8,}{lost:>22.1f}%   {tag}')

TRAIN_HARD = str(DATA_DIR / 'train_asr_hard.jsonl')
VAL_HARD   = str(DATA_DIR / 'val_asr_hard.jsonl')
TRAIN_LEG  = str(DATA_DIR / 'train_legacy.jsonl')
VAL_LEG    = str(DATA_DIR / 'val_legacy.jsonl')

!cp {DATA_DIR / 'train_asr_hard.jsonl'} {DATA_DRIVE / 'train_asr_hard_100k.jsonl'}
!cp {DATA_DIR / 'val_asr_hard.jsonl'}   {DATA_DRIVE / 'val_asr_hard_10k.jsonl'}
print('\nĐã copy bộ dữ liệu MỚI sang Drive — và đã khử trùng lặp nên không còn rò rỉ val↔train.')


In [ ]:
import zipfile

# Giai nen checkpoint Lan 4 TU TRÊN Drive làm DÒNG THAM CHIẾU so sánh.
# Chi lay weights + tokenizer o ngon goc, BO checkpoint-*/ (optimizer ~2.5 GB/epoch,
# suy luan khong can) -> tai ~1.6 GB thay vi 9.5 GB.
REF_ZIP = CKPT_ROOT / 'bmd1905_checkpoint.zip'
REF     = CKPT_ROOT / 'ref_lan4_legacy'
PREFIX  = 'checkpoints/bmd1905-sub-corrector/'

if (REF / 'config.json').exists():
    print(f'Đã có sẵn checkpoint tham chiếu: {REF}')
else:
    if not REF_ZIP.exists():
        have = sorted(p.name for p in CKPT_ROOT.iterdir())
        raise FileNotFoundError(
            f'Không thấy {REF_ZIP.name} trong {CKPT_ROOT}. Files hiện có: {have}'
        )
    REF.mkdir(parents=True, exist_ok=True)
    print(f'Đang giải nén {REF_ZIP.name} ({REF_ZIP.stat().st_size / 1024 ** 3:.2f} GB) ...')
    with zipfile.ZipFile(REF_ZIP) as z:
        all_names = z.namelist()
        picked = [n for n in all_names
                  if not n.endswith('/') and n.startswith(PREFIX) and '/checkpoint-' not in n]
        skipped = len([n for n in all_names if '/checkpoint-' in n])
        print(f'  Giu {len(picked)} file, bo {skipped} file checkpoint epoch trung gian')
        for n in picked:
            dest = REF / n[len(PREFIX):]
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(n) as src, open(dest, 'wb') as out:
                out.write(src.read())
    total = sum(f.stat().st_size for f in REF.rglob('*') if f.is_file()) / 1024 ** 3
    print(f'\nXong: {REF}  ({total:.2f} GB)')
    for f in sorted(REF.iterdir()):
        print(f'  {f.name:<34}{f.stat().st_size / 1024 ** 2:>10.1f} MB')

In [ ]:
import shutil
import subprocess
import sys as _sys

# ==== TẬP TEST VĂN HỌC — đo over-correction trên văn bản NGƯỜI VIẾT ====
# corpus/ KHÔNG nằm trong git (văn bản dịch có bản quyền) nên Colab phải kéo từ Drive.
# Cất ở Drive: vi-sub-correction/corpus/   (ngang hàng với input/ và output/)
CORPUS_DRIVE = DRIVE_ROOT / 'corpus'
CORPUS_LOCAL = REPO_DIR / 'corpus'

if CORPUS_DRIVE.exists() and any(CORPUS_DRIVE.glob('*.txt')):
    shutil.copytree(CORPUS_DRIVE, CORPUS_LOCAL, dirs_exist_ok=True)
    n = len(list(CORPUS_LOCAL.glob('*.txt')))
    print(f'Đã đồng bộ {n} tệp .txt từ {CORPUS_DRIVE}')

    # Sinh tập test: identity (đầu vào = đầu ra) + noisy (ngoài miền huấn luyện)
    subprocess.run([_sys.executable, '-m', 'src.data_processing.build_corpus_test',
                    '--out_dir', 'data/gold_test/control'], check=True)

    # Đo OVER-CORRECTION của mốc Lần 4 trên văn bản người viết.
    # Tập này đầu vào = đầu ra, nên MỌI thay đổi mô hình tạo ra đều là sửa nhầm.
    # Đây là phép đo mà tập 25 câu cũ không làm được (nó không có câu sạch nào).
    subprocess.run([_sys.executable, '-m', 'src.evaluation.run_benchmark',
                    '--test_file', 'data/gold_test/control/corpus_identity.jsonl',
                    '--model_name_or_path', str(REF),
                    '--output_report', str(REF / 'benchmark_corpus_identity.md'),
                    '--output_json', str(REF / 'metrics_corpus_identity.json'),
                    '--output_log', str(REF / 'benchmark_corpus_identity_log.txt'),
                    '--batch_size', '16'], check=True)
    print(f'\nĐọc {REF}/benchmark_corpus_identity.md để xem mô hình phá bao nhiêu câu đúng.')
else:
    print(f'CHƯA CÓ corpus trên Drive ({CORPUS_DRIVE}) — bỏ qua phần này.')
    print('  Upload thư mục corpus/ lên Drive nếu muốn đo over-correction.')
    print('  Bỏ qua cũng không ảnh hưởng các cell còn lại.')


In [ ]:
import json
from src.models.inference import Seq2SeqCorrector
from src.evaluation.metrics import evaluate_predictions

# GRID-SEARCH THAM SỐ GIẢI MÃ - KHÔNG phải train lại, chỉ mất vài phút.
# Đây là chỗ rẻ nhất để lấy thêm điểm: hiện repetition_penalty 1.2 VÀ no_repeat_ngram 3
# cùng phạt lặp, khá nặng tay, có thể buộc mô hình đổi từ không cần thiết.
#
# Tune trên VAL (asr_hard, 500 câu) - KHÔNG trên gold test 25 câu.
# Chọn cấu hình bằng chính tập test là overfit test, số liệu mất giá trị.
dev = [json.loads(l) for l in open(VAL_HARD, encoding='utf-8')][:500]
srcs = [r['noisy'] for r in dev]
refs = [r['clean'] for r in dev]
print(f'Grid-search trên {len(dev)} câu VAL (asr_hard). Gold test không bị đụng vào.\n')

GRID = [
    {'num_beams': 5, 'repetition_penalty': 1.20, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 5, 'repetition_penalty': 1.00, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 5, 'repetition_penalty': 1.05, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0},
    {'num_beams': 8, 'repetition_penalty': 1.00, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 4, 'repetition_penalty': 1.00, 'length_penalty': 1.2, 'no_repeat_ngram_size': 3},
]

decoder = Seq2SeqCorrector(model_name_or_path=str(REF), use_fallback=False)

hdr = f'{"beams":>6}{"rep_pen":>9}{"len_pen":>9}{"ngram":>7}{"EM%":>8}{"BLEU":>8}{"WER%":>8}{"CER%":>8}{"Ed-P%":>8}{"Ed-F1%":>9}'
print(hdr)
print('-' * len(hdr))
scored = []
for cfg in GRID:
    decoder.set_decoding(**cfg)
    preds = decoder.predict_batch(srcs, batch_size=32)
    m = evaluate_predictions(srcs, preds, refs)
    scored.append((m['exact_match'], m['edit_f1'], cfg, m))
    row = f'{cfg["num_beams"]:>6}{cfg["repetition_penalty"]:>9.2f}{cfg["length_penalty"]:>9.1f}' \
          f'{cfg["no_repeat_ngram_size"]:>7}{m["exact_match"]:>8.2f}{m["bleu"]:>8.2f}' \
          f'{m["wer"]:>8.2f}{m["cer"]:>8.2f}{m["edit_precision"]:>8.2f}{m["edit_f1"]:>9.2f}'
    print(row + ('   <- đang dùng' if cfg is GRID[0] else ''))

scored.sort(key=lambda t: (t[0], t[1]), reverse=True)
BEST_DECODING = scored[0][2]
BEAMS = BEST_DECODING['num_beams']
REP   = BEST_DECODING['repetition_penalty']
LENP  = BEST_DECODING['length_penalty']
NGRAM = BEST_DECODING['no_repeat_ngram_size']

print(f'\nCHỐT (dựa trên VAL, không đụng gold test): beams={BEAMS} rep_pen={REP} len_pen={LENP} ngram={NGRAM}')
print('Mọi benchmark phía dưới dùng ĐÚNG cấu hình này để so sánh công bằng.')

In [ ]:
import subprocess
import sys as _sys

# ==== BỘ KẾT QUẢ CƠ BẢN — đo mốc Lần 4 trên các tập test ĐÚNG BÀI TOÁN GỐC ====
# Tập 25 câu cũ quá nhỏ (mỗi câu = 4% chỉ số), nên ngoài nó ta đo thêm hai tập
# 200 câu sinh từ câu CHƯA dùng để train (anti-join theo nội dung), cùng dạng
# đầu vào nhưng đủ lớn để con số có ý nghĩa.
#
# Các tập test đã nằm sẵn trong repo (git pull là có), KHÔNG cần sinh lại trên Colab.
TEST_SUITES = [
    ('sample25',      'data/gold_test/gd1_basic/sample_test.jsonl',                 '25 câu gốc — để đối chiếu con số đã báo'),
    ('basic_unaccent','data/gold_test/gd1_basic/basic_test_unaccent_all.jsonl',     '200 câu mất trắng dấu — bài toán gốc'),
    ('basic_asrhard', 'data/gold_test/gd1_basic/basic_test_asr_hard.jsonl',         '200 câu nhiễu hỗn hợp'),
    ('corpus_identity','data/gold_test/control/corpus_identity.jsonl',            '962 câu sạch — đo over-correction'),
]

from pathlib import Path

for tag, path, desc in TEST_SUITES:
    if not Path(path).exists():
        print(f'[BỎ QUA] chưa có {path}')
        continue
    print('\n' + '=' * 74)
    print(f'  {tag}: {desc}')
    print('=' * 74)
    subprocess.run([_sys.executable, '-m', 'src.evaluation.run_benchmark',
                    '--test_file', path,
                    '--model_name_or_path', str(REF),
                    '--output_report', str(REF / f'benchmark_{tag}.md'),
                    '--output_json',   str(REF / f'metrics_{tag}.json'),
                    '--output_log',    str(REF / f'benchmark_{tag}_log.txt'),
                    '--batch_size', '16'], check=True)

print('\nXong. Bảng đối chiếu các tập test nằm trong từng metrics_*.json.')


In [ ]:
import json
from datetime import datetime

# Ghi manifest RIÊNG cho đợt Lan 4 để compare_runs đưa được VÀO bảng.
# Lan 4 train bằng script cũ nên không tự có manifest - các con số dưới CHÉP TỪ LOG
# Colab đã lưu trong notebooks/seq2seq_train_4_100k.ipynb, không phải số tự đặt.
(REF / 'manifest.json').write_text(json.dumps({
    'run_name': REF.name,
    'created_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    'ghi_chu': 'DIEN THU CONG tu log Lan 4 (khong do script tu sinh)',
    'git_commit': 'unknown (Lan 4 chay truoc khi co co che manifest)',
    'base_model': 'bmd1905/vietnamese-correction-v2',
    'device': 'cuda',
    'gpu': 'Tesla T4',
    'hyperparams': {
        'num_train_epochs': 3,
        'per_device_train_batch_size': 16,
        'learning_rate': 3e-05,
        'max_length': 128,
        'weight_decay': 0.01,
        'label_smoothing': 0.0,
        'padding': 'static',
        'fp16': True,
    },
    'dataset': {'train_size': 90000, 'val_size': 10000},
    'results': {
        'train_loss_final': 0.03736,
        'eval_loss_per_epoch': [0.02764, 0.01632, 0.01307],
        'best_eval_loss': 0.01307,
        'train_runtime_sec': 7941.0,
        'total_steps': 16875,
    },
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã ghi {REF / "manifest.json"}')

# Benchmark LẠI mốc tham chiếu bằng BỘ CHỈ SỐ MỚI (thêm CER / Exact Match / ROUGE-L)
# và ĐÚNG cấu hình giải mã vừa chốt -> đem so với đợt mới mới là so công bằng.
!python -m src.evaluation.run_benchmark \
    --test_file data/gold_test/gd1_basic/sample_test.jsonl \
    --model_name_or_path "{REF}" \
    --output_report "{REF / 'benchmark_lan5.md'}" \
    --output_log    "{REF / 'benchmark_log.txt'}" \
    --output_excel  "{REF / 'benchmark_results.xlsx'}" \
    --output_json   "{REF / 'metrics.json'}" \
    --batch_size 16 \
    --num_beams {BEAMS} \
    --repetition_penalty {REP} \
    --length_penalty {LENP} \
    --no_repeat_ngram_size {NGRAM}

In [ ]:
from src.models.train_seq2seq import resolve_output_dir

# ==== BIẾN THỂ CHÍNH ====
# Nhiễu khớp gold test (mất trắng dấu 70% + câu sạch 10%) + pad động + label smoothing
VARIANT  = 'asr_hard'
RUN_TAG  = 'lan5_asr_hard_ls01'
LABEL_SMOOTHING = 0.1

# CHỐNG TRÙNG (1/2): xem Drive đã có những đợt nào trước khi train
!python -m src.models.train_seq2seq --list_runs --output_dir "{CKPT_ROOT / 'bmd1905-sub-corrector'}"

# CHỐNG TRÙNG (2/2): tính tên thư mục đích TRƯỚC, in ra để mắt thấy.
# Tên đã tồn tại -> tự nhảy _v2, _v3. Không bao giờ đè đợt cũ.
DEST = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), RUN_TAG, overwrite=False)
print(f'\nDOT TRAIN NAY SE LUU VAO: {DEST}\n')

!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_HARD}' \
    --val_file   '{VAL_HARD}' \
    --output_dir '{DEST}' \
    --epochs 3 --batch_size 16 --lr 3e-5 \
    --label_smoothing {LABEL_SMOOTHING}

In [ ]:
# Benchmark đợt mới bằng ĐÚNG cấu hình giải mã đã chốt ở cell 4.
!python -m src.evaluation.run_benchmark \
    --test_file data/gold_test/gd1_basic/sample_test.jsonl \
    --model_name_or_path "{DEST}" \
    --output_report "{DEST / 'benchmark_lan5.md'}" \
    --output_log    "{DEST / 'benchmark_log.txt'}" \
    --output_excel  "{DEST / 'benchmark_results.xlsx'}" \
    --output_json   "{DEST / 'metrics.json'}" \
    --batch_size 16 \
    --num_beams {BEAMS} \
    --repetition_penalty {REP} \
    --length_penalty {LENP} \
    --no_repeat_ngram_size {NGRAM}

In [ ]:
# ĐẶT CẠNH NHAU: mốc Lan 4 vs đợt mới - cùng bộ chỉ số, cùng cách giải mã.
!python -m src.evaluation.compare_runs \
    --runs_root "{CKPT_ROOT}" \
    --output_md "{CKPT_ROOT / 'so_sanh_dot_train.md'}" \
    --sort_by exact_match

from google.colab import files
files.download(str(CKPT_ROOT / 'so_sanh_dot_train.md'))

In [ ]:
# ==== CELL TÙY CHỌN: tách riêng đóng góp của TỪNG thay đổi ====
# Đợt chính ở trên gộp 3 thay đổi (nhiễu + pad động + label smoothing), nên bảng so sánh
# chỉ cho biết TỔNG cải thiện, không nói cái nào đóng góp bao nhiêu. Hai biến thể dưới
# cô lập từng yếu tố. Mỗi đợt ~1-2 giờ; BỎ QUA nếu Colab sắp hết phiên.

from src.models.train_seq2seq import resolve_output_dir

# (B) asr_hard nhưng KHÔNG label smoothing -> cô lập tác động của LS
DEST_B = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), 'lan5_asr_hard_ls00', overwrite=False)
!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_HARD}' \
    --val_file   '{VAL_HARD}' \
    --output_dir '{DEST_B}' \
    --epochs 3 --batch_size 16 --lr 3e-5 --label_smoothing 0.0

# (C) nhiễu LEGACY (như Lan 4) nhưng CÓ pad động + LS -> cô lập tác động của NHIỄU
DEST_C = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), 'lan5_legacy_ls01', overwrite=False)
!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_LEG}' \
    --val_file   '{VAL_LEG}' \
    --output_dir '{DEST_C}' \
    --epochs 3 --batch_size 16 --lr 3e-5 --label_smoothing 0.1

In [ ]:
# Benchmark 2 biến thể phụ rồi mở lại bảng so sánh (cell 8) để đọc 4 dòng cạnh nhau.
for TAG, D in (('B: asr_hard + LS=0', DEST_B), ('C: legacy + LS=0.1', DEST_C)):
    print('=' * 70)
    print('BENCHMARK', TAG, '->', D)
    print('=' * 70)
    import subprocess, sys
    subprocess.run([
        sys.executable, '-m', 'src.evaluation.run_benchmark',
        '--test_file', 'data/gold_test/gd1_basic/sample_test.jsonl',
        '--model_name_or_path', str(D),
        '--output_json', str(D / 'metrics.json'),
        '--output_log', str(D / 'benchmark_log.txt'),
        '--batch_size', '16',
        '--num_beams', str(BEAMS),
        '--repetition_penalty', str(REP),
        '--length_penalty', str(LENP),
        '--no_repeat_ngram_size', str(NGRAM),
    ], check=True)

!python -m src.evaluation.compare_runs \
    --runs_root "{CKPT_ROOT}" \
    --output_md "{CKPT_ROOT / 'so_sanh_dot_train.md'}" \
    --sort_by exact_match